# Data Cleaning (Start From Here)

#### Preparation: Load data and examine sample size

In [32]:
# read the original anonymized dataframe
import pandas as pd
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/anonymized_berichten.xlsx")

In [4]:
len(set(df["UserGUID"].iloc[2:].to_list()))

856

In [5]:
len(df[["Message"]][2:]) # Number of messages = 33237

33237

In [6]:
len(df["Message"][df["richting (van of naar cliënt)"] == 'From client'])
# number of messages sent from patients = 13199

13199

In [33]:
# drop duplicated messages IF they are from the same user
df = df.drop_duplicates(subset=['UserGUID', 'Message'], keep='first')

In [34]:
len(df["Message"][df["richting (van of naar cliënt)"] == 'From client']) # Now there are 12754 messages left

12754

### Step 1. Remove html tags from the messages

In [35]:
# clean up the html tags
from bs4 import BeautifulSoup
def clean_html_with_soup(text):
    soup = BeautifulSoup(text, 'html.parser')
    return soup.get_text()

df['Message'] = df['Message'].apply(lambda x: clean_html_with_soup(str(x)))


In [36]:
df=df[2:]

In [37]:
# standardize column names
df.columns = ["provider_name", "program_name", "user_guid", "date_time", "message", "direction", "sender"]
# sort the messages by date time
df = df.sort_values("date_time")
# reset the index after sorting
df = df.reset_index(drop=True)
df.head()

,provider_name,program_name,user_guid,date_time,message,direction,sender
0,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-23 12:26:59.407,"Geachte ibd groep, Is mijn uitslag al binnen ...",From client,NaN
1,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-24 14:44:58.373,"Beste [PERSOON-1], Zojuist zag ik de uitslag ...",To client,mjong_mumc
2,MUMC MijnIBDcoach,MijnIBDcoach,12E3C542-9F96-43A4-BEE9-06225FEE170A,2014-09-27 07:46:52.670,Hallo! Vorige week is door [ZIEKENHUIS-1] [LOC...,From client,NaN
3,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-28 09:42:34.670,bloed in de ontlasting wordt steeds meer en st...,From client,NaN
4,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-29 10:08:41.710,"Beste [PERSOON-1], haaruitval kan te maken heb...",To client,htomlow_mumc


In [38]:
# check the head and tail for earliest and latest messages
df = df.reset_index(drop=True)
df["message_id"] = df.index + 1
df.head()

,provider_name,program_name,user_guid,date_time,message,direction,sender,message_id
0,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-23 12:26:59.407,"Geachte ibd groep, Is mijn uitslag al binnen ...",From client,NaN,1
1,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-24 14:44:58.373,"Beste [PERSOON-1], Zojuist zag ik de uitslag ...",To client,mjong_mumc,2
2,MUMC MijnIBDcoach,MijnIBDcoach,12E3C542-9F96-43A4-BEE9-06225FEE170A,2014-09-27 07:46:52.670,Hallo! Vorige week is door [ZIEKENHUIS-1] [LOC...,From client,NaN,3
3,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-28 09:42:34.670,bloed in de ontlasting wordt steeds meer en st...,From client,NaN,4
4,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-29 10:08:41.710,"Beste [PERSOON-1], haaruitval kan te maken heb...",To client,htomlow_mumc,5


### Step 2. Remove numbered placeholders
For example, [PERSOON-1] [PERSOON-2] are now all [PERSOON]

In [39]:
import re

# remove numbers of the place holders
def replace_numbered_patterns(text):
    text = re.sub(r'\[(\w+)-\d+\]', r'[\1]', text, flags=re.IGNORECASE) # replace numbered patterns 
    text = re.sub(r'\[person\]', '[PERSOON]', text, flags=re.IGNORECASE) # standardize the English [PERSON] with Dutch [PERSOON]
    text = re.sub(r'\[INSTITUTION\]', '[ZORGINSTELLING]', text, flags=re.IGNORECASE) # replace institutioin to zorginstelling
    text = re.sub(r':\)|:\(|;\)', '', text) # removes emojis
    text = re.sub(r"^[^\w([]+", '', text) # removes leading punctuations
    # Match any pattern of the form [WORD-<number>] and replace with [WORD]
    return text

# Apply the function
df['clean_message'] = df['message'].apply(lambda x: replace_numbered_patterns(str(x)))
df.head()

,provider_name,program_name,user_guid,date_time,message,direction,sender,message_id,clean_message
0,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-23 12:26:59.407,"Geachte ibd groep, Is mijn uitslag al binnen ...",From client,NaN,1,"Geachte ibd groep, Is mijn uitslag al binnen ..."
1,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-24 14:44:58.373,"Beste [PERSOON-1], Zojuist zag ik de uitslag ...",To client,mjong_mumc,2,"Beste [PERSOON], Zojuist zag ik de uitslag va..."
2,MUMC MijnIBDcoach,MijnIBDcoach,12E3C542-9F96-43A4-BEE9-06225FEE170A,2014-09-27 07:46:52.670,Hallo! Vorige week is door [ZIEKENHUIS-1] [LOC...,From client,NaN,3,Hallo! Vorige week is door [ZIEKENHUIS] [LOCAT...
3,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-28 09:42:34.670,bloed in de ontlasting wordt steeds meer en st...,From client,NaN,4,bloed in de ontlasting wordt steeds meer en st...
4,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-29 10:08:41.710,"Beste [PERSOON-1], haaruitval kan te maken heb...",To client,htomlow_mumc,5,"Beste [PERSOON], haaruitval kan te maken hebbe..."


### 3. Option to remove all placeholders

#### 3.1 Not remove all placeholders

In [40]:
df.head()

,provider_name,program_name,user_guid,date_time,message,direction,sender,message_id,clean_message
0,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-23 12:26:59.407,"Geachte ibd groep, Is mijn uitslag al binnen ...",From client,NaN,1,"Geachte ibd groep, Is mijn uitslag al binnen ..."
1,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-24 14:44:58.373,"Beste [PERSOON-1], Zojuist zag ik de uitslag ...",To client,mjong_mumc,2,"Beste [PERSOON], Zojuist zag ik de uitslag va..."
2,MUMC MijnIBDcoach,MijnIBDcoach,12E3C542-9F96-43A4-BEE9-06225FEE170A,2014-09-27 07:46:52.670,Hallo! Vorige week is door [ZIEKENHUIS-1] [LOC...,From client,NaN,3,Hallo! Vorige week is door [ZIEKENHUIS] [LOCAT...
3,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-28 09:42:34.670,bloed in de ontlasting wordt steeds meer en st...,From client,NaN,4,bloed in de ontlasting wordt steeds meer en st...
4,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-29 10:08:41.710,"Beste [PERSOON-1], haaruitval kan te maken heb...",To client,htomlow_mumc,5,"Beste [PERSOON], haaruitval kan te maken hebbe..."


#### 3.2 Optional: Completely remove all placeholders of sensitive information (Skip if you wish to keep them)

In [41]:
# it is optional to remove entities, as 
import re

def remove_entities(text): # notice that we did not remove anonymized entities such as [APOTHEEK] and [ZIEKENHUIS] which can be informative
    text = re.sub(r'\[persoon\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[locatie\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[url\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[telefoonnummer\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[emailadres\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[datum\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[ziekenhuis\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[apotheek\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[zorginstelling\]', '', text, flags=re.IGNORECASE)
    # remove words in the pattern of [word] but [hospital] and [pharmacy] are kept
    # remove punctuations again
    text = re.sub(r':\)|:\(|;\)', '', text)
    text = re.sub(r"^[^\w([]+", '', text)
    return text

df['clean_message_stripped'] = df['clean_message'].apply(lambda x: remove_entities(str(x)))

In [42]:
## clean up greetings and closings in the messages
import re

greetings_pattern = r"^\s*(beste|geachte|dear|hallo|hoi|hoi hoi|goedemorgen|goede morgen|goedemiddag|goede?n middag|goede?n avond|goedenavond|dag|hello|hi|morning|good morning|good afternoon|good evening)\s*[,!?.]*\s*"
signoffs_pattern = r"\b(best regards|regards|groe?t|groe?ten|gr|groetjes?|grtjs?|mvg|m?e?t?\s*vriendelijke?\s*gr\w*|met\s*vriendelijke?|hartelijke\s*gr\w*)\b\s*[,!?.]*\s*"

def remove_noise(text, pattern1, pattern2):
    # Remove greetings from the beginning
    cleaned_text = re.sub(pattern1, '', text, flags=re.IGNORECASE)
    # Remove signoffs from the end
    cleaned_text = re.sub(pattern2, '', cleaned_text, flags=re.IGNORECASE)
    # Strip any leading or trailing whitespace left over after substitutions

    return cleaned_text.strip()

# remove greetings and signoffs for both sets with or without placeholders
df['clean_message_stripped'] = df['clean_message_stripped'].apply(lambda x: remove_noise(str(x), greetings_pattern, signoffs_pattern))


In [43]:
df.head()

,provider_name,program_name,user_guid,date_time,message,direction,sender,message_id,clean_message,clean_message_stripped
0,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-23 12:26:59.407,"Geachte ibd groep, Is mijn uitslag al binnen ...",From client,NaN,1,"Geachte ibd groep, Is mijn uitslag al binnen ...","ibd groep, Is mijn uitslag al binnen van de b..."
1,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-24 14:44:58.373,"Beste [PERSOON-1], Zojuist zag ik de uitslag ...",To client,mjong_mumc,2,"Beste [PERSOON], Zojuist zag ik de uitslag va...","Zojuist zag ik de uitslag van de botscan, de b..."
2,MUMC MijnIBDcoach,MijnIBDcoach,12E3C542-9F96-43A4-BEE9-06225FEE170A,2014-09-27 07:46:52.670,Hallo! Vorige week is door [ZIEKENHUIS-1] [LOC...,From client,NaN,3,Hallo! Vorige week is door [ZIEKENHUIS] [LOCAT...,Vorige week is door mijn ontlasting onderzoc...
3,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-28 09:42:34.670,bloed in de ontlasting wordt steeds meer en st...,From client,NaN,4,bloed in de ontlasting wordt steeds meer en st...,bloed in de ontlasting wordt steeds meer en st...
4,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-29 10:08:41.710,"Beste [PERSOON-1], haaruitval kan te maken heb...",To client,htomlow_mumc,5,"Beste [PERSOON], haaruitval kan te maken hebbe...",haaruitval kan te maken hebben met medicatiege...


In [44]:
# save df
df.to_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/clean_message_data.xlsx")

In [5]:
import pandas as pd
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/clean_message_data.xlsx", index_col=0)
df.head()

,provider_name,program_name,user_guid,date_time,message,direction,sender,message_id,clean_message,clean_message_stripped
0,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-23 12:26:59.407,"Geachte ibd groep, Is mijn uitslag al binnen ...",From client,NaN,1,"Geachte ibd groep, Is mijn uitslag al binnen ...","ibd groep, Is mijn uitslag al binnen van de b..."
1,MUMC MijnIBDcoach,MijnIBDcoach,5422E6C6-75E8-4DAE-A6B6-E5EF46E2442E,2014-09-24 14:44:58.373,"Beste [PERSOON-1], Zojuist zag ik de uitslag ...",To client,mjong_mumc,2,"Beste [PERSOON], Zojuist zag ik de uitslag va...","Zojuist zag ik de uitslag van de botscan, de b..."
2,MUMC MijnIBDcoach,MijnIBDcoach,12E3C542-9F96-43A4-BEE9-06225FEE170A,2014-09-27 07:46:52.670,Hallo! Vorige week is door [ZIEKENHUIS-1] [LOC...,From client,NaN,3,Hallo! Vorige week is door [ZIEKENHUIS] [LOCAT...,Vorige week is door mijn ontlasting onderzoc...
3,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-28 09:42:34.670,bloed in de ontlasting wordt steeds meer en st...,From client,NaN,4,bloed in de ontlasting wordt steeds meer en st...,bloed in de ontlasting wordt steeds meer en st...
4,MUMC MijnIBDcoach,MijnIBDcoach,70E019C6-0F45-46D3-ADBF-16AD727C8EA0,2014-09-29 10:08:41.710,"Beste [PERSOON-1], haaruitval kan te maken heb...",To client,htomlow_mumc,5,"Beste [PERSOON], haaruitval kan te maken hebbe...",haaruitval kan te maken hebben met medicatiege...


In [7]:
na_rows = df[df['clean_message'].isna()]
na_rows

,provider_name,program_name,user_guid,date_time,message,direction,sender,message_id,clean_message,clean_message_stripped
741,MUMC MijnIBDcoach,MijnIBDcoach,92EBEE1B-5E04-4E80-B3A0-5148266F16E0,2015-05-18 20:01:31.737,.,From client,NaN,742,NaN,NaN
3878,MUMC MijnIBDcoach,MijnIBDcoach4.0,842AFFDD-ABA6-493B-8609-3BFCA670FE72,2017-12-13 18:10:35.450,????,From client,NaN,3879,NaN,NaN
4544,MUMC MijnIBDcoach,MijnIBDcoach4.0,9FA5CE1A-42E8-4014-9ABE-1A1810F90AAE,2018-03-06 09:02:17.217,??,From client,NaN,4545,NaN,NaN
4948,MUMC MijnIBDcoach,MijnIBDcoach4.0,25741F18-E56F-4278-BE5C-070AD131139E,2018-04-19 08:50:21.797,????,From client,NaN,4949,NaN,NaN
7879,MUMC MijnIBDcoach,MijnIBDcoach4.0,25741F18-E56F-4278-BE5C-070AD131139E,2019-01-21 14:58:50.737,??,From client,NaN,7880,NaN,NaN
28313,MUMC MijnIBDcoach,MijnIBDcoach5.0,D34DF484-C138-46AD-9291-4E5CC02727B6,2022-03-08 08:43:55.250,👌,From client,NaN,28314,NaN,NaN


### Step 5. Split messages into sentences 

In [32]:
# load nltk library
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [45]:
# using nltk instead of spacy
import nltk
from nltk.tokenize import sent_tokenize
import re

def split_sentence_with_matches(sentence, matches):
    
    
    parts = []
    for i in range(len(matches)):
        match = matches[i]
        part = sentence.split(match, 1)[0]
        
        if i > 0:
            part = "".join([matches[i-1], part])
        
        parts.append(part)
        remaining_part = sentence.split(match, 1)[1]
        sentence = remaining_part
    
    if remaining_part:
        part = "".join([matches[-1], remaining_part])
        parts.append(part)

    return parts


def split_sentence(sentence):
    pattern1 = r'(?<!^)Maar|En|Ik|Ze|Het'
    pattern2 = r'(?<!^)\.\s*[A-Z][a-z]+'
    matches1 = re.findall(pattern1, sentence)
    matches2 = re.findall(pattern2, sentence)
    matches = matches1
    if len(matches1) < len(matches2):
        matches = matches2
    smaller_sentences = []
    if matches:
        smaller_sentences = split_sentence_with_matches(sentence, matches)
    else:
        #print("Can't split the following sentence: ", "\n", sentence)
        pass
    return smaller_sentences


def nltk_split_into_sentences(df, column):
    sentence_data = []
    
    # Loop through each message in the original DataFrame
    for message_id, message in zip(df['message_id'], df[column]):
        sentences = sent_tokenize(message)
            
        # Append each sentence along with its existing Message_ID
        for sentence in sentences:
            
            sentence = sentence.strip()
            if len(sentence) <= 400 or 'voedinginformatie' in sentence.lower():
                sentence_data.append({'message_id': message_id, 'sentence': sentence})

            else:
                #print("\nOriginal: ", "\n", sentence, "\n")
                # Call split_sentence to get smaller sentences
                smaller_sentences = split_sentence(sentence)
                
                if smaller_sentences:                    
                    for smaller_sentence in smaller_sentences:
                        smaller_sentence = re.sub(r"^[,!.?\]]*\s*", "", smaller_sentence)

                        sentence_data.append({'message_id': message_id, 'sentence': smaller_sentence})

                else:
                    pass
            

    # Create a new DataFrame with the split sentences
    sentence_df = pd.DataFrame(sentence_data)

    return sentence_df

# Example usage
df_fromclient = df[df["direction"] == "From client"]
sentence_df_nltk = nltk_split_into_sentences(df_fromclient, column="clean_message")

In [47]:
# examine the split
pd.set_option("max_colwidth", 400)
sentence_df = sentence_df_nltk
sentence_df.head()

,message_id,sentence
0,1,"Geachte ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?"
1,1,Groetjes [PERSOON] [PERSOON]
2,3,Hallo!
3,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar [ZIEKENHUIS] gestuurd.
4,3,Graag zou ik de uitkomst hiervan vernemen.


### Step 4. Merge incorrectly split sentences
Some sentences were split where it shouldn't be split, so we need to merge them back together

In [48]:
# find sentences ending in ""
abbreviation_pattern = r"\b[a-zA-Z]\.[a-zA-Z]\.[a-zA-Z]\.\s*$"
abbreviation_pattern_2 = r"incl\.\s*$"

In [49]:
sentence_df = sentence_df.dropna(subset=['sentence'])
sentence_df

# now there are 53,705 rows

,message_id,sentence
0,1,"Geachte ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?"
1,1,Groetjes [PERSOON] [PERSOON]
2,3,Hallo!
3,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar [ZIEKENHUIS] gestuurd.
4,3,Graag zou ik de uitkomst hiervan vernemen.
...,...,...
53700,32686,Wat is normaal.
53701,32686,Om de hoeveel tijd vinden jullie dat ik bloed moet laten prikken en ontlasting moet inleveren?
53702,32687,"Hallo [PERSOON] Gisteren, laat in de middag, potje en formulieren, ontvangen Vanochtend naar [ZIEKENHUIS] gebracht Er stond geen fax nummer op, daar zouden ze zelf achter aan gaan."
53703,32687,"Heb nog veel en waterige diaree Ben dus benieuwd naar de uitslag, laat jij me die weten ?"


In [50]:
to_drop = []
for i, row in sentence_df[:-1].iterrows():
    sentence = row["sentence"]
    if re.search(abbreviation_pattern, sentence, flags=re.IGNORECASE):
        sentence_df.loc[i, "sentence"] += " " + sentence_df.loc[i + 1, "sentence"]

        to_drop.append(i + 1)

# Drop merged sentences from DataFrame
sentence_df.drop(to_drop, inplace=True)

# Reset index
sentence_df.reset_index(drop=True, inplace=True)

In [51]:
# examine the sentence df after merging
sentence_df.head()

,message_id,sentence
0,1,"Geachte ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?"
1,1,Groetjes [PERSOON] [PERSOON]
2,3,Hallo!
3,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar [ZIEKENHUIS] gestuurd.
4,3,Graag zou ik de uitkomst hiervan vernemen.


In [52]:
sentence_df.tail()
# Now there are 32687 sentences

,message_id,sentence
53258,32686,Wat is normaal.
53259,32686,Om de hoeveel tijd vinden jullie dat ik bloed moet laten prikken en ontlasting moet inleveren?
53260,32687,"Hallo [PERSOON] Gisteren, laat in de middag, potje en formulieren, ontvangen Vanochtend naar [ZIEKENHUIS] gebracht Er stond geen fax nummer op, daar zouden ze zelf achter aan gaan."
53261,32687,"Heb nog veel en waterige diaree Ben dus benieuwd naar de uitslag, laat jij me die weten ?"
53262,32687,Groetjes [PERSOON] [PERSOON]


### Step 5: Clean the sentences in the same way as messages

In [53]:
# remove leading punctuations
sentence_df['clean_sentence'] = sentence_df['sentence'].apply(lambda x: remove_punctuations(str(x)))
# replace entities 
sentence_df['clean_sentence'] = sentence_df['clean_sentence'].apply(lambda x: remove_entities(str(x)))
# remove noise for both sets with/without placeholder
sentence_df['clean_sentence'] = sentence_df['clean_sentence'].apply(lambda x: remove_noise(str(x), greetings_pattern, signoffs_pattern))

In [54]:
sentence_df.head()

,message_id,sentence,clean_sentence
0,1,"Geachte ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?","ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?"
1,1,Groetjes [PERSOON] [PERSOON],
2,3,Hallo!,
3,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar [ZIEKENHUIS] gestuurd.,Vorige week is door mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar gestuurd.
4,3,Graag zou ik de uitkomst hiervan vernemen.,Graag zou ik de uitkomst hiervan vernemen.


In [55]:
# save the sentence df before removing short sentences

sentence_df.to_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/sentence_data.xlsx")

In [35]:
# one more time print <15 char clean_sentence
sentence_df[sentence_df["clean_sentence"].str.len() <= 15]

,message_id,sentence,clean_sentence_placeholder,clean_sentence
1,1,Groetjes [PERSOON] [PERSOON],[PERSOON] [PERSOON],
2,3,Hallo!,,
5,3,Bedankt!,Bedankt!,Bedankt!
8,4,?,,
10,7,"Groeten, [PERSOON]",[PERSOON],
...,...,...,...,...
53274,32683,Dankjewel!,Dankjewel!,Dankjewel!
53276,32683,Bedankt alvast!,Bedankt alvast!,Bedankt alvast!
53279,32685,"Met vriendelijke groet, [PERSOON]",[PERSOON],
53285,32686,Wat is normaal.,Wat is normaal.,Wat is normaal.


In [56]:
sentence_df = sentence_df[sentence_df["clean_sentence"].str.len() > 15]
sentence_df

,message_id,sentence,clean_sentence
0,1,"Geachte ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?","ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?"
3,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar [ZIEKENHUIS] gestuurd.,Vorige week is door mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar gestuurd.
4,3,Graag zou ik de uitkomst hiervan vernemen.,Graag zou ik de uitkomst hiervan vernemen.
6,4,"bloed in de ontlasting wordt steeds meer en steeds frequenter, ook wordt haarverlies intensiever.","bloed in de ontlasting wordt steeds meer en steeds frequenter, ook wordt haarverlies intensiever."
7,4,Ligt dit aan de medicatie?,Ligt dit aan de medicatie?
...,...,...,...
53256,32686,Volgens mij is dat in februari geweest.,Volgens mij is dat in februari geweest.
53257,32686,[LOCATIE] een lange tijd terug.,een lange tijd terug.
53259,32686,Om de hoeveel tijd vinden jullie dat ik bloed moet laten prikken en ontlasting moet inleveren?,Om de hoeveel tijd vinden jullie dat ik bloed moet laten prikken en ontlasting moet inleveren?
53260,32687,"Hallo [PERSOON] Gisteren, laat in de middag, potje en formulieren, ontvangen Vanochtend naar [ZIEKENHUIS] gebracht Er stond geen fax nummer op, daar zouden ze zelf achter aan gaan.","Gisteren, laat in de middag, potje en formulieren, ontvangen Vanochtend naar gebracht Er stond geen fax nummer op, daar zouden ze zelf achter aan gaan."


In [57]:
sentence_df = sentence_df.reset_index(drop=True)
sentence_df["sentence_id"] = sentence_df.index + 1

In [58]:
sentence_df.head()

,message_id,sentence,clean_sentence,sentence_id
0,1,"Geachte ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?","ibd groep, Is mijn uitslag al binnen van de botscan van afgelopen donderdag?",1
1,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar [ZIEKENHUIS] gestuurd.,Vorige week is door mijn ontlasting onderzocht om CRP wat beter in beeld te krijgen en resultaten zijn ahgi naar gestuurd.,2
2,3,Graag zou ik de uitkomst hiervan vernemen.,Graag zou ik de uitkomst hiervan vernemen.,3
3,4,"bloed in de ontlasting wordt steeds meer en steeds frequenter, ook wordt haarverlies intensiever.","bloed in de ontlasting wordt steeds meer en steeds frequenter, ook wordt haarverlies intensiever.",4
4,4,Ligt dit aan de medicatie?,Ligt dit aan de medicatie?,5


In [59]:
len(sentence_df)

41119

In [60]:
sentence_df.to_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/sentence_data_for_analysis.xlsx")

### Now run the translate_sentences script and then come back

In [60]:
# read the translated df
translated_sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data.xlsx", index_col=0)
translated_sentence_df

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ikben2wekengeledenmetspoedopgenomeninhetinmetv...,I was rushed into the [PRESSION] two weeks ago...,1
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...",Ikkreegacuuteenpijnlijkedrukopdeborstschouderb...,I was acutely under a painful pressure on the ...,2
2,3,Het begon 1 uur na het avondeten.,Hetbegon1uurnahetavondeten,It started 1 hour after dinner.,3
3,3,"Ik had al de hele dag migraine, had dus ook we...",Ikhadaldeheledagmigrainehaddusookweiniggegeten,"I had migraines all day, so I hadn't eaten much.",4
4,3,"Ik werd heel erg misselijk, braakneigingen, du...",Ikwerdheelergmisselijkbraakneigingenduizeligen...,"I got very nauseous, vomiting, dizzy and rejuv...",5
...,...,...,...,...,...
41263,32679,"Ze hebben de urine op kweek gezet, kan morgenv...",Zehebbendeurineopkweekgezetkanmorgenvroegdeuit...,"They've put the urine on culture, can get the ...",41264
41264,32679,Afgelopen jaren is er al vaker bloed in mijn u...,Afgelopenjareniseralvakerbloedinmijnurinegevon...,"In recent years, blood has been found in my ur...",41265
41265,32679,Wat adviseert u hiermee te doen?,Watadviseertuhiermeetedoen,What do you suggest you do with this?,41266
41266,32681,"Hoi [PERSOON], Oke, dat is iig al een gerustst...",Okedatisiigaleengeruststellingikwachtdeuitslag...,"Hi [PRESSON], Okay, that's a relief;) I'll wai...",41267


In [61]:
import pandas as pd
# Reset index to ensure alignment before comparison
sentence_df = sentence_df.reset_index(drop=True)
translated_sentence_df = translated_sentence_df.reset_index(drop=True)

# Find the first index where the two DataFrames differ
for i, (sent1, sent2) in enumerate(zip(sentence_df["Sentence"], translated_sentence_df["Sentence"])):
    if sent1 != sent2:
        print(f"First difference at index: {i}")
        break
else:
    print("No differences found!")


First difference at index: 659


In [77]:
pd.set_option("max_colwidth", None)
sentence_df[657:661]

,Message_ID,Sentence,Clean_Sentence
657,437,Verleden keer heb ik niet meegedaan omdat ik toch bij jullie onder behandeling ben.,Verleden keer heb ik niet meegedaan omdat ik toch bij jullie onder behandeling ben.
658,437,Wat is eigenlijk jullie advies t.b.v. jullie patiënten?,Wat is eigenlijk jullie advies t.b.v. jullie patiënten?
659,440,Bedankt voor de reactie; Ik laat t.z.t. ondertussen heb ik uitslag van het BVO ontvangen: dat is gelukkig gunstig.,Bedankt voor de reactie; Ik laat t.z.t. ondertussen heb ik uitslag van het BVO ontvangen: dat is gelukkig gunstig.
660,443,"[PERSOON], ik ben weer aan mijn laatste potje purinethol toe.",ik ben weer aan mijn laatste potje purinethol toe.


In [76]:
pd.set_option("max_colwidth", None)
translated_sentence_df[657:661]

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID
657,437,Verleden keer heb ik niet meegedaan omdat ik toch bij jullie onder behandeling ben.,Verledenkeerhebiknietmeegedaanomdatiktochbijjullieonderbehandelingben,Last time I didn't participate because I'm being treated with you guys.,658
658,437,Wat is eigenlijk jullie advies t.b.v. jullie patiënten?,Watiseigenlijkjullieadviestbvjulliepatinten,What exactly is your advice to your patients?,659
659,437,jullie patiënten?,julliepatinten,Your patients?,660
660,440,Bedankt voor de reactie; Ik laat t.z.t. ondertussen heb ik uitslag van het BVO ontvangen: dat is gelukkig gunstig.,BedanktvoordereactieIklaattztondertussenhebikuitslagvanhetBVOontvangendatisgelukkiggunstig,Thanks for the comment; I am late to receive results from the BVO in the meantime: that is fortunately beneficial.,661


In [78]:

translated_sentence_df.drop(index=659, inplace=True)
translated_sentence_df[657:661]

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID
657,437,Verleden keer heb ik niet meegedaan omdat ik toch bij jullie onder behandeling ben.,Verledenkeerhebiknietmeegedaanomdatiktochbijjullieonderbehandelingben,Last time I didn't participate because I'm being treated with you guys.,658
658,437,Wat is eigenlijk jullie advies t.b.v. jullie patiënten?,Watiseigenlijkjullieadviestbvjulliepatinten,What exactly is your advice to your patients?,659
660,440,Bedankt voor de reactie; Ik laat t.z.t. ondertussen heb ik uitslag van het BVO ontvangen: dat is gelukkig gunstig.,BedanktvoordereactieIklaattztondertussenhebikuitslagvanhetBVOontvangendatisgelukkiggunstig,Thanks for the comment; I am late to receive results from the BVO in the meantime: that is fortunately beneficial.,661
661,443,"[PERSOON], ik ben weer aan mijn laatste potje purinethol toe.",ikbenweeraanmijnlaatstepotjepurinetholtoe,"[PERSON], I'm back to my last jar of purinethol.",662


In [79]:
clean_sentences = sentence_df["Clean_Sentence"].to_list()
len(clean_sentences)

41267

In [80]:
translated_sentence_df["Clean_Sentence"] = clean_sentences

In [26]:
# Find sentences <15 characters
translated_sentence_df[translated_sentence_df["Clean_Sentence"].str.len() < 15]

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID


In [81]:
translated_sentence_df

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID
0,3,Ik ben 2 weken geleden met spoed opgenomen in het [PERSOON] in [LOCATIE] met verdenking van hartklachten.,Ik ben 2 weken geleden met spoed opgenomen in het in met verdenking van hartklachten.,I was rushed into the [PRESSION] two weeks ago in [LOCATION] with suspicion of heart problems.,1
1,3,"Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.","Ik kreeg acuut een pijnlijke druk op de borst, schouderbladen, uitstralend naar de armen.","I was acutely under a painful pressure on the chest, shoulder blades, radiating to the arms.",2
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,It started 1 hour after dinner.,3
3,3,"Ik had al de hele dag migraine, had dus ook weinig gegeten.","Ik had al de hele dag migraine, had dus ook weinig gegeten.","I had migraines all day, so I hadn't eaten much.",4
4,3,"Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.","Ik werd heel erg misselijk, braakneigingen, duizelig en oprispingen.En vreselijke pijn.","I got very nauseous, vomiting, dizzy and rejuvenating, and terrible pain.",5
...,...,...,...,...,...
41263,32679,"Ze hebben de urine op kweek gezet, kan morgenvroeg de uitslag krijgen hiervan.","Ze hebben de urine op kweek gezet, kan morgenvroeg de uitslag krijgen hiervan.","They've put the urine on culture, can get the results of this in the morning.",41264
41264,32679,Afgelopen jaren is er al vaker bloed in mijn urine gevonden en ik vraag me af of het iets te maken heeft met de ziekte v Crohn of een fist oid?,Afgelopen jaren is er al vaker bloed in mijn urine gevonden en ik vraag me af of het iets te maken heeft met de ziekte v Crohn of een fist oid?,"In recent years, blood has been found in my urine and I wonder if it has anything to do with the disease v Crohn or a fist oid?",41265
41265,32679,Wat adviseert u hiermee te doen?,Wat adviseert u hiermee te doen?,What do you suggest you do with this?,41266
41266,32681,"Hoi [PERSOON], Oke, dat is iig al een geruststelling ;) ik wacht de uitslag even af verder.","Oke, dat is iig al een geruststelling ;) ik wacht de uitslag even af verder.","Hi [PRESSON], Okay, that's a relief;) I'll wait for the results.",41267


In [83]:
# Find rows where 'Column1' does NOT start with a letter
filtered_df = translated_sentence_df[~translated_sentence_df["Clean_Sentence"].str.match(r'^[A-Za-z0-9]', na=False)]

filtered_df[50: 100]

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID
1393,1091,"(ik ben met vakantie eind juli tot en met 12 augustus, dus met voldoende voorraad overbrug ik dat zonder problemen).","(ik ben met vakantie eind juli tot en met 12 augustus, dus met voldoende voorraad overbrug ik dat zonder problemen).","(I am on holiday at the end of July until 12 August, so with sufficient stock I bridge it without any problems).",1394
1402,1097,"Beste [PERSOON]/[PERSOON]/[PERSOON], Ik heb twee vragen.","//, Ik heb twee vragen.","Dear [PERSON]/[PERSON]/[PERSON], I have two questions.",1403
1415,1097,"(mocht de poli-afspraak verschoven worden naar een andere datum dan liefst pas na 11.15 uur) groet, HUM [PERSOON] [DATUM] [TELEFOONNUMMER]",(mocht de poli-afspraak verschoven worden naar een andere datum dan liefst pas na 11.15 uur) HUM,(If the policy appointment is postponed to another date than preferably after 11.15 p.m.) greets HUM [PRESSON] [DATE] [PHONE NUMBER],1416
1416,1099,"Beste [PERSOON]/[PERSOON]/[PERSOON], Eind juli vertrek ik met vakantie naar [LOCATIE].","//, Eind juli vertrek ik met vakantie naar .","Dear [PERSOON]/[PERSOON]/[PERSOON], At the end of July I leave for [LOCATION].",1417
1419,1101,"Beste [PERSOON]/[PERSOON]/[PERSOON], Ik heb vragen over mijn medicijngebruik met betrekking tot zwangerschap: Zijn er risico's gezien mijn medicijngebruik Prednisolon en Vedolizumab als mijn vriendin zwanger zou worden?","//, Ik heb vragen over mijn medicijngebruik met betrekking tot zwangerschap: Zijn er risico's gezien mijn medicijngebruik Prednisolon en Vedolizumab als mijn vriendin zwanger zou worden?","Dear [PERSOON]/[PERSOON]/[PERSOON], I have questions about my drug use regarding pregnancy: Are there any risks considering my drug use Prednisolone and Vedolizumab if my girlfriend would become pregnant?",1420
1421,1102,"Beste [PERSOON]/[PERSOON]/[PERSOON], Meestal volgt er een snelle reactie van uw kant als ik een vraag stel via ibd-coach, dit keer niet, wellicht vakantie?","//, Meestal volgt er een snelle reactie van uw kant als ik een vraag stel via ibd-coach, dit keer niet, wellicht vakantie?","Dear [PERSOON]/[PERSOON]/[PERSOON], is there usually a quick response from your side when I ask a question via ibd coach, not this time, perhaps vacation?",1422
1430,1103,"(met zelfde tijdstip als de 21e) groet, HUM [PERSOON] [DATUM] [TELEFOONNUMMER]",(met zelfde tijdstip als de 21e) HUM,(at the same time as the 21st) greets HUM [PRESSON] [DATE] [PHONE NUMBER],1431
1431,1105,"Beste [PERSOON] / [PERSOON] / [PERSOON], Luisterend naar mijn lichaam is het verstandiger om een langzamer afbouwtempo prednisolon aan te houden.","/ / , Luisterend naar mijn lichaam is het verstandiger om een langzamer afbouwtempo prednisolon aan te houden.","Dear [PERSON] / [PERSON] / [PERSON], Listening to my body it is wiser to maintain a slower phasing-out tempo prednisolone.",1432
1436,1107,"Beste [PERSOON] / [PERSOON] / [PERSOON], De huisarts heeft een oogzalf voorgeschreven vanwege een dik ooglid, dit ter voorkoming van ontsteking.","/ / , De huisarts heeft een oogzalf voorgeschreven vanwege een dik ooglid, dit ter voorkoming van ontsteking.","Dear [PERSOON] / [PERSOON] / [PERSOON], The GP prescribed an eye ointment because of a thick eyelid, this to prevent inflammation.",1437
1440,1109,"Beste [PERSOON]/[PERSOON]/[PERSOON], Ik ben bijna door mijn voorraad Prednisolon (15 mg per dag) en door mijn voorraad TacalD3 kauwtabletten heen.","//, Ik ben bijna door mijn voorraad Prednisolon (15 mg per dag) en door mijn voorraad TacalD3 kauwtabletten heen.","Best [PERSOON]/[PERSOON]/[PERSOON], I'm almost out of stock Prednisolone (15 mg per day) and through my supply of TacalD3 chewable tablets.",1441


In [33]:
translated_sentence_df.to_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")